In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         File EXTRACTOOOOOOOR        #
#######################################

# Define target website
URL = "https://www.ipos.gov.sg/about-ip/plant-variety-rights/practice-guidelines-circulars"
# Define the website domain
domain = "https://www.ipos.gov.sg"
# Define staging site for redirections report
staging = "https://staging.d9vo48leqc1gd.amplifyapp.com/"

# Define variables
files = {}
folder_path = "docs"
hrefSchema = {}
# Define custom file path after /files (e.g. "/folder1/folder2")
# Output -> "/files/folder1/folder2/document.pdf"
docsFolderPath = ""

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
except:
  pass

def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL)
  soup = BeautifulSoup(page.content, "html.parser")

  # 1. Loop through all <a> tags found in HTML
  # 2. Attempt to detect '/docs/' for each href link in <a> tag
  # 3. Format file name
  # 4. Build dictionary with extracted data
  # 5. Initiate downloadFiles() function once 1-4 is completed
  for i in soup.find_all(['a']):
    href = i.get('href')
    # print(href)
    if href and '/docs/' in i['href'] or href and 'go.gov.sg' in i['href']:

      # Build dictionary for easier search
      # f-strings -> To embed variables directly to strings
      # fileName -> Dictionary Comprehension
      # Count -> Unique identifier for access
      files[count] = {"Text": i.text, "File name": i['href'][:i['href'].find('?')].split("/")[-1], "Download Link": i['href'] if 'go.gov.sg' in i['href'] else domain + i['href'] if domain not in i['href'] else i['href'], "Staging site": staging + f"files{docsFolderPath if True else None}/{i['href'][:i['href'].find('?')].split("/")[-1]}", "Updated link": i['href'][:i['href'].find('?')], "Original link": i['href']}
      r = requests.head(files[count]["Download Link"], allow_redirects=True)
      files[count]["Download Link"] = r.url.split('%')[0]
      files[count]["File name"] = r.url[:r.url.find('?')].split("/")[-1] if '?' in r.url else r.url.split("/")[-1]
      files[count]["File name"] = files[count]["File name"].split('%')[0]

      print(files[count]["Download Link"], files[count]["File name"])

      count += 1
  downloadFiles()

def downloadFiles():
  # Create 'docs' folder if it doesnt exists on Google Colab workspace
  if not os.path.exists(folder_path):
      os.makedirs(folder_path)

  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # 1. Loops through each dictionary item
  # 2. Retrieve a file's download link
  # 3. Define the file's intended file path and name
  # 4. Copy the file and store in its intended file path and name
  # 5. Zip the folder once 1-4 is completed
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download Link']

        # Define file path and name
        file_path = os.path.join(f"{folder_path}", f"{files[index]['File name']}")
        
        # Copy file from download link
        urllib.request.urlretrieve(url, file_path)

    except Exception as e:
        pass

  # 1. Loops through each dictionary item
  # 2. Calculate file size in bytes
  # 3. Calculate file size based on bytes to use between kB/MB/GB
  # 4. Build json schema for hyperlinks
  # 5. Print json schema
  for index, value in enumerate(files):
    try:
        # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
        fileSize = str(math.ceil(os.path.getsize(f"docs/{files[index]['File name']}")/1024))
    
        # Calculate and define file size
        # Removed temporarily - str(int(fileSize)/1000000) + ' GB'
        fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'
    
        # Build with hardcoded json schema
        # f-strings -> To embed variables directly to strings
        hrefSchema = {
              "type": "text",
              "marks": [
                {
                  "type": "link",
                  "attrs": {
                    "href": f"/files{docsFolderPath if True else None}/{files[index]['File name']}"
                    }
                }
              ],
              # Original File Name is used as we want to display Circular 1 instead of circular-1
              "text": f"{files[index]['Text']} [{'DOCX' if '.docx' in files[index]['Download Link'] else 'DOC' if '.doc' in files[index]['Download Link'] else 'XLXS' if '.xlxs' in files[index]['Download Link'] else 'XLS' if '.xls' in files[index]['Download Link'] else 'ZIP' if '.zip' in files[index]['Download Link'] else 'PDF'}, {fileSize}]"
            }
            
        print(json.dumps(hrefSchema))
    except Exception as e:
      pass

getDictionary(URL)

df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("file-extractor-report.csv", index=False)

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         Image EXTRACTOOOOOOOR       #
#######################################

# Define target website
URL = "https://www.clc.gov.sg/events/lectures/view/creating-liveable-cities-through-car-lite-urban-mobility-launch"
# Define the website domain
domain = "https://www.clc.gov.sg"
# Define staging site for redirections report
staging = "https://staging.d9vo48leqc1gd.amplifyapp.com/"

# Define variables
files = {}
folder_path = "images"
hrefSchema = {}
# Define custom file path after /images (e.g. "/folder1/folder2")
# Output -> "/images/folder1/folder2/document.png"
docsFolderPath = ""

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
except:
  pass

def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL)
  soup = BeautifulSoup(page.content, "html.parser")

  # 1. Loop through all <a> tags found in HTML
  # 2. Attempt to detect '/docs/' for each href link in <a> tag
  # 3. Format file name
  # 4. Build dictionary with extracted data
  # 5. Initiate downloadFiles() function once 1-4 is completed
  for i in soup.find_all(['img']):
    try:    
        # Build dictionary for easier search
        # f-strings -> To embed variables directly to strings
        # fileName -> Dictionary Comprehension
        # Count -> Unique identifier for access
        files[count] = {"Original image name": i['alt'], "Updated image name": i['src'][:i['src'].find('?')].split("/")[-1], "Download link": domain + i['src'], "Staging link": staging + f"images/{docsFolderPath if True else None}{i['src'][:i['src'].find('?')].split("/")[-1]}"}
        print(files[count]["Download link"], files[count]["Original image name"])
        count += 1
        
    except Exception as e:
        pass

  downloadFiles()

def downloadFiles():
  # Create 'docs' folder if it doesnt exists on Google Colab workspace
  if not os.path.exists(folder_path):
      os.makedirs(folder_path)

  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # 1. Loops through each dictionary item
  # 2. Retrieve an image's download link
  # 3. Define the image's intended file path and name
  # 4. Copy the image and store in its intended file path and name
  # 5. Zip the folder once 1-4 is completed
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download link']

        # Define file path and name
        file_path = os.path.join(f"{folder_path}", f"{files[index]['Updated image name']}")
        
        # Copy file from download link
        urllib.request.urlretrieve(url, file_path)

        hrefSchema = {
            "type": "image",
            "src": f"/images{docsFolderPath if True else None}/{files[index]['Updated image name']}",
            "alt": ""
        }
        print(json.dumps(hrefSchema))
    except Exception as e:
        pass

getDictionary(URL)

df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("image-extractor-report.csv", index=False)

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math

#######################################
#     Link Extractooooor (Website)    #
#######################################

# Define target website
url = "https://www.clc.gov.sg/events/lectures"
domain = "https://www.clc.gov.sg"

def getDictionary(link):
  count = 0

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(link)
  soup = BeautifulSoup(page.content, "html.parser")
  soup = soup.find("div", class_="sf_colsIn sf_2cols_1in_75")

  # 1. Loop through all <a> tags found in HTML
  # 2. Build dictionary with extracted data
  # 3. Initiate downloadFiles() function once 1-4 is completed
  for i in soup.find_all(['a']):
    print(i)
    try:
        # Build dictionary for easier search
        # f-strings -> To embed variables directly to strings
        # fileName -> Dictionary Comprehension
        # Count -> Unique identifier for access
        files[count] = {"Text": i.text, "Link": i['alt']}
        count += 1
    except Exception as e:
        pass

getDictionary(url)

In [ ]:
import pandas as pd
import json
import urllib.request
import requests
from bs4 import BeautifulSoup

########################################
#       Category Extractoooor (CSV)    #
########################################

url = "https://www.agc.gov.sg/newsroom/media-releases/newsitem/"

# Read the CSV and store in variable
df = pd.read_csv('agc.csv')
df_copy = df.copy()

# Create new Category column and assign NULL value to each row
df_copy['Category'] = None

# Loop through each row in CSV
for index, value in enumerate(df_copy['Title']):
    # Send a GET request to fetch the page HTML
    page = requests.get(f"{url}{df_copy['Original page name'][index]}")
    soup = BeautifulSoup(page.content, "html.parser")
    # Find all <article> tag with class "news-single--content mb60"
    soup = soup.find("article", class_="news-single--content mb60")
    # Store category of each link into "Category" column
    df_copy.loc[index, "Category"] = soup.find('h1').text

df_copy.to_csv('agc.csv', index=False, header=True)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup
import requests
import re
import math

########################################
#     HTML Extractoooor (Link, CLC)    #
########################################

domain = "https://www.clc.gov.sg"
URL = "https://www.clc.gov.sg/research-publications/publications/better-cities/GetDataView//"
count = 0
data = {}

for number in range (0,5):
    tempLinks = []
    # Send a POST request to get all links in each repsective year
    page  = requests.post(URL, data = {'DivName':f'articleList202{number}', 'Year':f'202{number}'})
    soup = BeautifulSoup(page.content, "html.parser")

    # Extract all links in each year and append to tempLinks list
    for i in soup.find_all(['a']):
        href = i.get('href')
        if "/" in href and href not in tempLinks:
            tempLinks.append(href)

    # Build dictionary for each item in tempLinks list
    for i in tempLinks:
        # Build data dictionary for scrapping and exporting
        data[count] = {"Year": f"202{number}", "url": "https://www.clc.gov.sg" + i, "path": i}
        count+=1

# Loop through data dictionary
for i in range(0, len(data)):
    # Send a GET request to fetch the page HTML
    page = requests.get(data[i]["url"])
    soup = BeautifulSoup(page.content, "html.parser")

    # Find all <div> tag with the class "sf_2cols_1in_75"
    content = soup.find("div", class_="sf_2cols_1in_75")
    # Find all <div> tag with the class "row" from content
    content1 = content.find_all("div", class_="row")
    # Create a new column named "HTML" and store the HTML 
    data[i]["html"] = content1

df = pd.DataFrame.from_dict(data, orient='index')
df.to_csv("better-cities.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup
import requests
import re
import math

########################################
#    HTML Extractoooor (Link, MUIS)    #
########################################

domain = "https://www.muis.gov.sg"
URL = "https://www.muis.gov.sg/Media/Media-Releases"
count = 0
pageData = {}
date = []
pageDate = []

# Send a GET request to fetch the page HTML
page = requests.get(URL)
soup = BeautifulSoup(page.content, "html.parser")
# Find all <div> tag with the class "panel-body"
content = soup.find_all("div", class_="panel-body")

# Loop through each content
for i in content:
    # Extract the date for each content
    for dates in re.findall(r"\b\d{2} [A-Za-z]{3} \d{4}\b", str(i)):
        date.append(dates)
    # Extract the page link for each content
    for y, v in enumerate(str(content).split("<br/>")):
        link = BeautifulSoup(v, "html.parser")
        link = link.find_all("a")
        for x in link:
            pageData[y - 1] = {"URL": domain + x["href"]}

# Build pageData dictionary with date
for i, v in enumerate(date):
    pageData[i]["date"] = v

# Loop through each pageData and fetch the HTML code
for i, v in enumerate(pageData):
    page = requests.get(pageData[i]["URL"])
    soup = BeautifulSoup(page.content, "html.parser")
    
    content = soup.find("div", class_="col-md-8 left-content")
    header = content.find("h1")
    pageDate.append(content.find("p").text)

    # Create a new column named "HTML" and store the HTML 
    pageData[i]["HTML"] = content
    # Create a new column named "Title" and store the Title 
    pageData[i]["Title"] = header.text

df = pd.DataFrame.from_dict(pageData, orient='index')
df.to_csv("muis-media-releases.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, REACH)   #
########################################

# Declare dataset as global variable 
df = pd.read_csv('public-consultation-processed.csv')
df_copy = df.copy()
pageData = {}

# df_copy["Title"] = df_copy["URL"].apply(lambda url: url.split("/")[-1])

# Loop through each URL in df_copy
for index, value in enumerate(df_copy['URL']):
    # Send a GET request to fetch the page HTML
    page = requests.get(df_copy['URL'][index])
    soup = BeautifulSoup(page.content, "html.parser")

    title = value.split("/")[-1].replace("-", " ").upper()  
    header = soup.find("div", class_="description-group")
    agency = header.find("img")["alt"]
    date = header.find("dl", class_="description-list class").find("dd").text.split("-")[0][:-1]
    content = soup.find_all("div", class_="accordion accordion--consultation")

    # Find all <div> tag with the class "mt-5"
    try:
        pdf = soup.find_all("div", class_="mt-5")[1]
    except Exception as e:
        pdf = "No PDF"

    # Build the dictionary for exporting
    pageData[index] = {"Title": title, "Date": date, "Category": agency, "Status": "", "URL": value, "header": header, "content": content, "pdf": pdf}

export = pd.DataFrame.from_dict(pageData, orient='index')
export.to_csv("public-consultation-processed.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, CLC)     #
########################################

# Declare dataset
df = pd.read_csv('clc-digital-library.csv')
df_copy = df.copy()

pageData = []

# Loop through each URL in the csv file
for index, value in enumerate(df_copy['URL']):
    # Fetch site's HTML
    page = requests.get(df_copy['URL'][index])
    soup = BeautifulSoup(page.content, "html.parser")
    
    # Pinpoint the exact "Container" that hosts the content we want
    date = soup.find("span", class_="date")
    date = date.text.split(" ")
    pageData.append(("1 " + date[0][:3] + " " + date[1]))

# Insert pageData list to a new column called "date"
df_copy["date"] = pageData

# Export the dataframe into csv file
df_copy.to_csv("clc-digital-library.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, REACH)   #
########################################

# Declare dataset
df = pd.read_csv('reach-automation.csv')
df_copy = df.copy()

pageData = []

# Loop through each URL in the csv file
for index, value in enumerate(df_copy['Site link']):
    # Fetch site's HTML
    page = requests.get(df_copy['Site link'][index])
    soup = BeautifulSoup(page.content, "html.parser")
    
    # Pinpoint the exact "Container" that hosts the content we want
    content = soup.find_all("div", class_="sfContentBlock sf-Long-text a-rich-text")
    index = len(content)-1 if '<div class="sfContentBlock sf-Long-text a-rich-text"></div>' not in str(content[len(content)-1]) else len(content)-2
    pageData.append(content[index])

# Insert pageData list to a new column called "HTML"
df_copy["HTML"] = pageData

# Export the dataframe into csv file
df_copy.to_csv("automation-processed.csv", index=False)

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (API, CSA)     #
########################################

# URL of the API
url = "https://www.csa.gov.sg/api/card-title/GetNewsCardTitles"
domain = "https://www.csa.gov.sg"
pageData = {}

form_data = {
    # Advisories
    # "additionalFilters": '{"__msdisposeindex":221,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":228},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"ba30433d-afed-4a80-a4f9-5c382eb59dcc","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":229},"Name":"alert","_itemPathSeparator":"_","__msdisposeindex":230},{"IsGroup":false,"Ordinal":1,"Join":"OR","ItemPath":"_0_1","Value":"e6302e05-16ad-41ff-9724-2e69857d5744","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":231},"Name":"advisory0f71fe1b-b89b-415e-b160-2a32433bf027","_itemPathSeparator":"_","__msdisposeindex":232},{"IsGroup":false,"Ordinal":2,"Join":"OR","ItemPath":"_0_2","Value":"e94c5117-4324-4d51-bd5a-4bce7d62a443","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":233},"Name":"advisories","_itemPathSeparator":"_","__msdisposeindex":234}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # Alerts
    # "additionalFilters": '{"__msdisposeindex":221,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":226},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"a7eb5a8f-42c2-4ccd-b1c9-b5d4f6c783ca","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":227},"Name":"alerts","_itemPathSeparator":"_","__msdisposeindex":228},{"IsGroup":false,"Ordinal":1,"Join":"OR","ItemPath":"_0_1","Value":"4c690993-74e1-4de3-9652-1960cc8114f9","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":229},"Name":"alertsc3632aa9-41c4-47f9-94b5-180f66eb538f","_itemPathSeparator":"_","__msdisposeindex":230}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # Press Release
    # "additionalFilters": '{"__msdisposeindex":235,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":242},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"d8c63703-63f6-4cee-9ee8-8a1508ad73a2","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":243},"Name":"press-releases","_itemPathSeparator":"_","__msdisposeindex":244}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # Speeches
    # "additionalFilters": '{"__msdisposeindex":247,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":{"FieldName":null,"FieldType":null,"Operator":null,"__msdisposeindex":249},"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":248},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"be4e3d69-dcde-4d05-96e7-f51e42581629","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":251},"Name":"speeches","_itemPathSeparator":"_","__msdisposeindex":250}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # News
    "additionalFilters": '{"__msdisposeindex":255,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":{"FieldName":null,"FieldType":null,"Operator":null,"__msdisposeindex":257},"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":256},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"a3c0f106-4aff-4705-8ac4-7f4d9ba46b15","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":259},"Name":"news-articles","_itemPathSeparator":"_","__msdisposeindex":258}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    "limit": "1000",
    "page": "1",
    "selectedItemsIds": "",
    "selectionMode": "FilteredItems"
}

# Make a POST request to API
response = requests.post(url, data=form_data)
# Check for response status
response.raise_for_status()
# Convert response to JSON
json_response = json.loads(response.text)

# Loop through each response and prepare dictionary for scrapping and exporting
for index, value in enumerate(json_response["objects"]):
    pageData[index] = {"Title": value["title"], "Date": value["date"], "URL": domain + value["link"], "Category": value["tag"]}

# Loop through each item in pageData dictionary
for i, v in enumerate(pageData):
    print(pageData[i]["Title"])

    # Make a GET request to fetch the page HTML
    page = requests.get(pageData[i]["URL"])
    soup = BeautifulSoup(page.content, "html.parser")
    # Find all <div> tags with the class "sfContentBlock sf-Long-text a-rich-text"
    content = soup.find_all("div", class_ = "sfContentBlock sf-Long-text a-rich-text")  
    # Create a new column named "HTML" and store the HTML 
    pageData[i]["HTML"] = content

(pd.DataFrame.from_dict(data=pageData, orient='index')
   .to_csv('csa-news.csv', header=True))

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup
import threading

########################################
#        To Do - Multithreading        #
########################################

# URL of the API
url = "https://www.csa.gov.sg/api/card-title/GetNewsCardTitles"
domain = "https://www.csa.gov.sg"
pageData = {}

form_data = {
    # Advisories
    "additionalFilters": '{"__msdisposeindex":221,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":228},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"ba30433d-afed-4a80-a4f9-5c382eb59dcc","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":229},"Name":"alert","_itemPathSeparator":"_","__msdisposeindex":230},{"IsGroup":false,"Ordinal":1,"Join":"OR","ItemPath":"_0_1","Value":"e6302e05-16ad-41ff-9724-2e69857d5744","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":231},"Name":"advisory0f71fe1b-b89b-415e-b160-2a32433bf027","_itemPathSeparator":"_","__msdisposeindex":232},{"IsGroup":false,"Ordinal":2,"Join":"OR","ItemPath":"_0_2","Value":"e94c5117-4324-4d51-bd5a-4bce7d62a443","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":233},"Name":"advisories","_itemPathSeparator":"_","__msdisposeindex":234}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    "limit": "1000",
    "page": "1",
    "selectedItemsIds": "",
    "selectionMode": "FilteredItems"
}
    
response = requests.post(url, data=form_data)
response.raise_for_status()  # Check if the request was successful

json_response = json.loads(response.text)

for index, value in enumerate(json_response["objects"]):
    pageData[index] = {"Title": value["title"], "Date": value["date"], "URL": domain + value["link"], "Category": value["tag"], "Description": value["desc"]}

pageDataList = list(pageData.items())
firstHalf = pageDataList[:len(pageDataList)//2]
secondHalf = pageDataList[len(pageDataList)//2:]

def multiThreading(half):
    for i, v in enumerate(half):
        print(i, v)
        # print(pageData[i]["Title"])
        # page = requests.get(pageData[i]["URL"])
        # soup = BeautifulSoup(page.content, "html.parser")
        # content = soup.find("div", class_ = "sfContentBlock sf-Long-text a-rich-text")  
        # pageData[i]["HTML"] = content

thread1 = threading.Thread(target=process_part, args=(firstHalf, "First Half"))
thread2 = threading.Thread(target=process_part, args=(secondHalf, "Second Half"))

thread1.start()
thread2.start()

thread1.join()
thread2.join()
# (pd.DataFrame.from_dict(data=pageData, orient='index')
#    .to_csv('csa-advisories.csv', header=True))

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (API, CLC)     #
########################################

# URL of the API
url = "https://www.clc.gov.sg/research-publications/publications/digital-library/Search/"
domain = "https://www.clc.gov.sg"
pageData = {}

counter = 1
pageCounter = 0
while counter < 19:
    form_data = {
        "page": f"{counter}"    
    }

    # Make a POST request to API
    response = requests.post(url, data=form_data)
    # Check for response status
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    # Find all <div> tag with class "digital-library-item-result"
    content = soup.find_all("div", class_ = "digital-library-item-result")
    
    for i, v in enumerate(content):
        try:
            title = v.find("h4", class_ = "title")
            type = v.find("span", class_ = "publication-type")
            publication = v.find("span", class_ = "publication-type-hidden")
            blurb = v.find("p", class_ = "blurb")
            # Build dictionary for exporting
            pageData[pageCounter] = {"Title": title.text, "URL": domain + title.find("a").get('href'), "Publication type": type.text, "Category": publication.text, "Subtitle": blurb.text}
        except Exception as e:
            # Build dictionary for exporting
            pageData[pageCounter] = {"Title": title.text, "URL": domain + title.find("a").get('href'), "Publication type": None, "Category": None, "Subtitle": blurb.text}

        pageCounter += 1
    counter += 1

# 
for i, v in enumerate(pageData):
    # Make a POST request to fetch page
    response = requests.post(pageData[i]["URL"])
    # Check for response status
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    # Find all <div> tag with class "sf_colsIn sf_2cols_1in_67"
    content = soup.find_all("div", class_ = "sf_colsIn sf_2cols_1in_67")
    # Create a new column named "HTML" and store the HTML 
    pageData[i]["HTML"] = content

(pd.DataFrame.from_dict(data=pageData, orient='index')
   .to_csv('clc-digital-library.csv', header=True))

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup
import re

########################################
#     HTML Extractoooor (API, MUIS)    #
########################################

# URL of the API
url = "https://www.muis.gov.sg/officeofthemufti/Khutbah"
yearList = ["2024", "2023", "2022", "2021", "2020"]
languageList = ["493dd241-c9dc-48e7-af8f-34aa2a07ce10", "f3dba80c-5b3b-4866-8dec-b9ac0fb02b59", "57fc2908-8a6b-41fa-8ed2-f93771dc1bcb"]

prayerDict = {}

# Loop though all year in yearList
for year in yearList:
    # For each year, loop through each language
    for language in languageList:
        titleList = []
        try:
            request_body = {
                "scAction": "GetKhubatArticles",
                "scController": "PageContent",
                "searchData[GUID]": f"{language}",
                "searchData[PageSize]": "1000",
                "searchData[Page]": "1",
                "searchData[Year]": f"{year}"
            }
            headers = {'Content-Type': 'application/x-www-form-urlencoded'}
            # Make a POST request to API
            response = requests.post(url, data=request_body, headers=headers)

            # Check for response status
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            date = soup.find_all("p", class_="italic")
            
            dateList = []
            contentListEng = []

            # Format all dates and append to dateList
            for i in date:
                dateFormatted = i.text.replace("\r", "").replace("\n", "")
                dateList.append(dateFormatted[:list(re.finditer(r'\d+', dateFormatted))[-1].end()])
            
            content = soup.find_all("div", class_="row")
            title = soup.find_all("p", class_="strong")

            # Append title to titleList
            for i in title:
                titleList.append(i.text)

            # Append all content found to contentListEng
            while True:
                # Break if all content are appended to contentListEng
                if len(content) == 0:
                    break
                contentListEng.append(str(content[0]) + str(content[1]) if len(content) % 2 == 0 else str(content[0]))
                try:
                    # Remove the appended content from content and repeat till its empty
                    del content[0]
                except Exception as e:
                    pass
                try:
                    del content[0]
                except Exception as e:
                    pass

            # Build dictionary for exporting
            for i, v in enumerate(dateList):
                # If date dont exist, create a new nested dictionary
                if v not in prayerDict:
                    prayerDict[v] = {}
                try:
                    if language == "f3dba80c-5b3b-4866-8dec-b9ac0fb02b59":
                        prayerDict[v]["Malay Title"] = titleList[i]
                    else:
                        prayerDict[v]["Title"] = titleList[i]
                    prayerDict[v][language] = contentListEng[i]
                except Exception as e:
                    prayerDict[v][language] = ""
                    
        except Exception as e:
            print(year, language)
            continue
                    
print(prayerDict)

(pd.DataFrame.from_dict(data=prayerDict, orient='index')
   .to_csv('muis.csv', header=True))